# E01 · Homogeneous transforms, quaternions and gimbal lock

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/06_estimation_perception/E01_homogeneous_transforms_quaternions_and_gimbal_lo.ipynb)

**Track:** Estimation and perception · **Skill axis (sheet):** Estimation and perception · **Time:** about 55 min

**Skill Hub concepts:** Homogeneous transform; Quaternion and Euler angles

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("E01")

## Map: one 4x4 matrix moves points between frames

$$ {}^{A}T_{B} = \begin{bmatrix} R & p \\ 0 & 1 \end{bmatrix}, \qquad {}^{A}x = {}^{A}T_{B}\,{}^{B}x, \qquad
{}^{A}T_{C} = {}^{A}T_{B}\,{}^{B}T_{C}, \qquad ({}^{A}T_{B})^{-1} = \begin{bmatrix} R^\top & -R^\top p \\ 0 & 1 \end{bmatrix}$$

Read ${}^{A}T_{B}$ as "pose of B expressed in A". Chains compose left to right exactly like tf2 (lab ROS05).

| Representation | Numbers | Good | Bad |
|---|---|---|---|
| Rotation matrix | 9 | composes by multiplication, no singularities | 6 constraints to keep orthonormal |
| Euler angles (ZYX yaw-pitch-roll) | 3 | readable | **gimbal lock** at pitch = +-90 degrees (yaw and roll become the same axis) |
| Axis-angle | 3 or 4 | intuitive | awkward to compose |
| Unit quaternion $(x, y, z, w)$ (ROS order) | 4 | no singularity, cheap to compose and interpolate | double cover: $q$ and $-q$ are the same rotation |

Quaternion from axis $\hat u$ and angle $\theta$: $q = (\hat u \sin\tfrac{\theta}{2},\ \cos\tfrac{\theta}{2})$.
Product (Hamilton), with $q = (\mathbf{v}, w)$: $q_1 q_2 = (w_1\mathbf{v}_2 + w_2\mathbf{v}_1 + \mathbf{v}_1 \times \mathbf{v}_2,\ w_1 w_2 - \mathbf{v}_1\cdot\mathbf{v}_2)$.

In [ ]:
import numpy as np

## Task 1: build, compose and invert transforms

In [ ]:
def rot_z(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1.0]])

def make_T(R, p):
    raise NotImplementedError  # TODO

def inv_T(T):
    raise NotImplementedError  # TODO

def apply(T, x):
    raise NotImplementedError  # TODO

lab.check("transforms", make_T, inv_T, apply)

## Task 2: quaternions (x, y, z, w)

In [ ]:
def quat_from_axis_angle(axis, theta):
    raise NotImplementedError  # TODO

def quat_mul(q1, q2):
    raise NotImplementedError  # TODO

def quat_to_R(q):
    raise NotImplementedError  # TODO

lab.check("quaternions", quat_from_axis_angle, quat_mul, quat_to_R)

## Task 3: ZYX Euler angles and where they break
Return `(yaw, pitch, roll)` with $R = R_z(\text{yaw})\,R_y(\text{pitch})\,R_x(\text{roll})$.

In [ ]:
def euler_zyx(R):
    raise NotImplementedError  # TODO

lab.check("euler", euler_zyx, quat_to_R)

Gimbal lock: at pitch = 90 degrees, changing yaw or roll produces the same rotation family. Two different
(yaw, roll) pairs give the same matrix:

In [ ]:
try:
    def R_zyx(y, p, r):
        Ry = np.array([[np.cos(p), 0, np.sin(p)], [0, 1, 0], [-np.sin(p), 0, np.cos(p)]])
        Rx = np.array([[1, 0, 0], [0, np.cos(r), -np.sin(r)], [0, np.sin(r), np.cos(r)]])
        return rot_z(y) @ Ry @ Rx
    A, B = R_zyx(0.3, np.pi / 2, 0.0), R_zyx(0.0, np.pi / 2, -0.3)
    print("same matrix:", np.allclose(A, B), " -> one degree of freedom is lost")
except NotImplementedError:
    pass

## Checkpoint: Math
**M1 (Apply).** Frame B is rotated 90 degrees about z and translated (2, 0, 0) in A. Point (1, 0, 0) in B is where in A?
Answer as a list `[x, y, z]`.

**M2 (Apply).** The `w` of the quaternion for a 120-degree rotation about any axis?

**M3 (Remember).** Degrees of freedom lost at gimbal lock?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)
lab.answer("M3", None)

## Checkpoint: Concept
**K1 (Understand).** Your IMU filter outputs `q` and the next step outputs `-q`. Problem?
- a) the attitude flipped 180 degrees
- b) none for the rotation itself (q and -q are the same rotation), but interpolation and averaging must pick the same hemisphere
- c) the quaternion is not normalized

**K2 (Analyze).** Explain why robot software stores orientation as quaternions or matrices and converts to Euler angles
only for display, using the gimbal lock demo above.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `e01.h`).
3. In `roboprep/checks/e01.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()